# Local CPU Performance Benchmark — LCC-FASD

Notebook này **chỉ dùng để benchmark hiệu năng chạy local**. Không train 50 epoch, không đánh giá chất lượng FAS. Mục tiêu là đo throughput CNN trên CPU và ảnh hưởng của batch size, DataLoader workers, CPU threads, input resolution và augmentation.


## 2. Kiểm tra phần cứng

In [1]:
%pip install --upgrade pip
%pip install pillow numpy pandas tqdm psutil

Note: you may need to restart the kernel to use updated packages.
  Using cached pandas-3.0.5-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached tzdata-2026.4-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.5-cp312-cp312-win_amd64.whl (9.8 MB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)
Using cached tzdata-2026.4-py2.py3-none-any.whl (347 kB)

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [tqdm]
   ------------- -------------------------- 1/3 [tqdm]
   ------------- -------------------------- 1/3 [tqdm]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
   -------------------------- ------------- 2/3 [pandas]
 

In [2]:
import os, gc, time, platform
from pathlib import Path
import torch, torchvision, pandas as pd, psutil
from PIL import Image
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn as nn

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("OS:", platform.platform())
print("CPU logical processors:", os.cpu_count())
print("RAM total (GB):", round(psutil.virtual_memory().total/(1024**3),2))
print("PyTorch CPU threads:", torch.get_num_threads())
print("CUDA available:", torch.cuda.is_available())
print("XPU available:", torch.xpu.is_available())
print("XPU device count:", torch.xpu.device_count())
print("XPU device:", torch.xpu.get_device_name(0) if torch.xpu.is_available() else "N/A")
print("Device used: XPU")

Python: 3.12.3
PyTorch: 2.14.0+xpu
Torchvision: 0.29.0+xpu
OS: Windows-11-10.0.26200-SP0
CPU logical processors: 14
RAM total (GB): 15.43
PyTorch CPU threads: 12
CUDA available: False
XPU available: True
XPU device count: 1
XPU device: Intel(R) Graphics
Device used: XPU


## 3. Kiểm tra dataset

In [4]:
DATASET_ROOT=Path("./dataset")
if not DATASET_ROOT.exists(): raise FileNotFoundError(f"Không tìm thấy {DATASET_ROOT.resolve()}")
print("Dataset root:", DATASET_ROOT.resolve())

def find_class_dirs(root):
    out=[]
    for p in root.rglob("*"):
        if p.is_dir():
            names={x.name.lower() for x in p.iterdir() if x.is_dir()}
            if "real" in names and "spoof" in names: out.append(p)
    return out
class_roots=find_class_dirs(DATASET_ROOT)
print("Các thư mục có real/spoof:")
for p in class_roots: print(" -",p)
if not class_roots: raise RuntimeError("Không tìm thấy thư mục có real và spoof.")

def score(p):
    n=p.name.lower(); return (10 if "train" in n else 0)+(5 if "training" in n else 0)
TRAIN_ROOT=sorted(class_roots,key=score,reverse=True)[0]
print("TRAIN_ROOT:",TRAIN_ROOT.resolve())
preview=ImageFolder(str(TRAIN_ROOT))
print("Classes:",preview.classes)
print("Mapping:",preview.class_to_idx)
print("Training images:",len(preview))


Dataset root: C:\Users\YOGA\Desktop\Class\05_Deep_Neural_Networks\project-01\face-anti-spoofing\dataset
Các thư mục có real/spoof:
 - dataset\LCC_FASD\LCC_FASD_development
 - dataset\LCC_FASD\LCC_FASD_evaluation
 - dataset\LCC_FASD\LCC_FASD_training
TRAIN_ROOT: C:\Users\YOGA\Desktop\Class\05_Deep_Neural_Networks\project-01\face-anti-spoofing\dataset\LCC_FASD\LCC_FASD_training
Classes: ['real', 'spoof']
Mapping: {'real': 0, 'spoof': 1}
Training images: 8299


## 4. CNN benchmark
Model cố định giữa các phép đo; đây không phải model cuối cùng.

In [5]:
class BenchmarkCNN(nn.Module):
    def __init__(self,num_classes=2):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(inplace=True),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(inplace=True),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(inplace=True),nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(inplace=True),nn.MaxPool2d(2))
        self.pool=nn.AdaptiveAvgPool2d((1,1)); self.classifier=nn.Linear(256,num_classes)
    def forward(self,x):
        x=self.pool(self.features(x)); return self.classifier(torch.flatten(x,1))
print(BenchmarkCNN())


BenchmarkCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNor

## 5. Hàm benchmark
Mỗi cấu hình có warm-up rồi đo một số batch; xuất `images/sec` và thời gian epoch ước lượng.


In [8]:
DEVICE = torch.device("xpu")

def make_loader(batch_size,num_workers,img_size,use_augmentation=True):
    if use_augmentation:
        tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.RandomHorizontalFlip(),transforms.ColorJitter(brightness=.10,contrast=.10,saturation=.10,hue=.02),transforms.ToTensor()])
    else:
        tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.ToTensor()])
    ds=ImageFolder(str(TRAIN_ROOT),transform=tf)
    kw=dict(batch_size=batch_size,shuffle=True,num_workers=num_workers,pin_memory=False,drop_last=True)
    if num_workers>0: kw["persistent_workers"]=True
    return ds,DataLoader(ds,**kw)

def benchmark_config(batch_size,num_workers,img_size,cpu_threads,warmup_batches=5,measure_batches=30,use_augmentation=True):
    gc.collect(); torch.set_num_threads(cpu_threads)
    ds,loader=make_loader(batch_size,num_workers,img_size,use_augmentation)
    model=BenchmarkCNN().to(DEVICE).train(); opt=torch.optim.AdamW(model.parameters(),lr=3e-4); crit=nn.CrossEntropyLoss()
    it=iter(loader)
    for _ in range(warmup_batches):
        try: x,y=next(it)
        except StopIteration: it=iter(loader); x,y=next(it)
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        opt.zero_grad(set_to_none=True); loss=crit(model(x),y); loss.backward(); opt.step()
    times=[]
    for _ in range(measure_batches):
        try: x,y=next(it)
        except StopIteration: it=iter(loader); x,y=next(it)
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        torch.xpu.synchronize()
        t = time.perf_counter()

        opt.zero_grad(set_to_none=True)
        loss = crit(model(x), y)
        loss.backward()
        opt.step()

        torch.xpu.synchronize()
        times.append(time.perf_counter() - t)
    sec=sum(times)/len(times); ips=batch_size/sec
    return dict(batch_size=batch_size,num_workers=num_workers,img_size=img_size,cpu_threads=cpu_threads,dataset_images=len(ds),seconds_per_batch=sec,images_per_sec=ips,estimated_epoch_min=len(ds)/ips/60)


In [10]:
result_xpu_b64 = benchmark_config(
    batch_size=64,
    num_workers=2,
    img_size=128,
    cpu_threads=8,
    warmup_batches=10,
    measure_batches=100,
    use_augmentation=True
)

print(result_xpu_b64)

{'batch_size': 64, 'num_workers': 2, 'img_size': 128, 'cpu_threads': 8, 'dataset_images': 8299, 'seconds_per_batch': 0.27987553200004184, 'images_per_sec': 228.6730802890995, 'estimated_epoch_min': 0.6048664166844655}


## 6. Benchmark A — batch size × workers
Giữ input 128 và CPU threads = 8. Test batch 16/32/64/128 × workers 0/2/4.


In [6]:
results_A=[]
CPU_THREADS_A=min(8,os.cpu_count() or 1)
for batch in [16,32,64,128]:
    for workers in [0,2,4]:
        print(f"Running batch={batch}, workers={workers}, threads={CPU_THREADS_A}")
        try:
            r=benchmark_config(batch,workers,128,CPU_THREADS_A); results_A.append(r)
            print(f"  {r["images_per_sec"]:.2f} images/s | {r["estimated_epoch_min"]:.2f} min/epoch")
        except Exception as e: print("  FAILED:",repr(e))
df_A=pd.DataFrame(results_A)
display(df_A.sort_values("images_per_sec",ascending=False))


Running batch=16, workers=0, threads=8
  84.12 images/s | 1.64 min/epoch
Running batch=16, workers=2, threads=8
  92.21 images/s | 1.50 min/epoch
Running batch=16, workers=4, threads=8
  87.14 images/s | 1.59 min/epoch
Running batch=32, workers=0, threads=8
  87.17 images/s | 1.59 min/epoch
Running batch=32, workers=2, threads=8
  73.65 images/s | 1.88 min/epoch
Running batch=32, workers=4, threads=8
  78.91 images/s | 1.75 min/epoch
Running batch=64, workers=0, threads=8
  82.64 images/s | 1.67 min/epoch
Running batch=64, workers=2, threads=8
  90.28 images/s | 1.53 min/epoch
Running batch=64, workers=4, threads=8
  66.07 images/s | 2.09 min/epoch
Running batch=128, workers=0, threads=8
  90.58 images/s | 1.53 min/epoch
Running batch=128, workers=2, threads=8
  82.77 images/s | 1.67 min/epoch
Running batch=128, workers=4, threads=8
  79.15 images/s | 1.75 min/epoch


,batch_size,num_workers,img_size,cpu_threads,dataset_images,seconds_per_batch,images_per_sec,estimated_epoch_min
1,16,2,128,8,8299,0.173521,92.208007,1.500051
9,128,0,128,8,8299,1.413059,90.583632,1.526950
7,64,2,128,8,8299,0.708900,90.280731,1.532073
3,32,0,128,8,8299,0.367083,87.173722,1.586678
2,16,4,128,8,8299,0.183623,87.135142,1.587381
0,16,0,128,8,8299,0.190206,84.119257,1.644293
10,128,2,128,8,8299,1.546403,82.772758,1.671041
6,64,0,128,8,8299,0.774483,82.635756,1.673811
11,128,4,128,8,8299,1.617134,79.152381,1.747473
5,32,4,128,8,8299,0.405508,78.913336,1.752767


## 7. Benchmark B — CPU threads
Giữ cấu hình nhanh nhất từ A; test 4/8/12/14 threads (giới hạn theo số logical CPU thực tế).


In [7]:
if len(df_A)==0: raise RuntimeError("Benchmark A chưa có kết quả")
best_A=df_A.iloc[df_A["images_per_sec"].idxmax()]
BEST_BATCH=int(best_A.batch_size); BEST_WORKERS=int(best_A.num_workers)
thread_candidates=sorted(set(t for t in [4,8,12,14] if t <= (os.cpu_count() or 1)))
results_B=[]
for threads in thread_candidates:
    print(f"Running batch={BEST_BATCH}, workers={BEST_WORKERS}, threads={threads}")
    try:
        r=benchmark_config(BEST_BATCH,BEST_WORKERS,128,threads); results_B.append(r)
        print(f"  {r["images_per_sec"]:.2f} images/s | {r["estimated_epoch_min"]:.2f} min/epoch")
    except Exception as e: print("  FAILED:",repr(e))
df_B=pd.DataFrame(results_B); display(df_B.sort_values("images_per_sec",ascending=False))


Running batch=16, workers=2, threads=4
  60.55 images/s | 2.28 min/epoch
Running batch=16, workers=2, threads=8
  82.44 images/s | 1.68 min/epoch
Running batch=16, workers=2, threads=12
  78.48 images/s | 1.76 min/epoch
Running batch=16, workers=2, threads=14
  72.91 images/s | 1.90 min/epoch


,batch_size,num_workers,img_size,cpu_threads,dataset_images,seconds_per_batch,images_per_sec,estimated_epoch_min
1,16,2,128,8,8299,0.194080,82.440235,1.677781
2,16,2,128,12,8299,0.203876,78.479117,1.762465
3,16,2,128,14,8299,0.219450,72.909459,1.897102
0,16,2,128,4,8299,0.264265,60.545342,2.284514


## 8. Benchmark C — input resolution
Giữ batch/workers/threads tốt nhất; test 96, 128, 160.


In [8]:
best_B=df_B.iloc[df_B["images_per_sec"].idxmax()] if len(df_B) else best_A
BEST_THREADS=int(best_B.cpu_threads)
results_C=[]
for size in [96,128,160]:
    print(f"Running size={size}, batch={BEST_BATCH}, workers={BEST_WORKERS}, threads={BEST_THREADS}")
    try:
        r=benchmark_config(BEST_BATCH,BEST_WORKERS,size,BEST_THREADS); results_C.append(r)
        print(f"  {r["images_per_sec"]:.2f} images/s | {r["estimated_epoch_min"]:.2f} min/epoch")
    except Exception as e: print("  FAILED:",repr(e))
df_C=pd.DataFrame(results_C); display(df_C.sort_values("images_per_sec",ascending=False))


Running size=96, batch=16, workers=2, threads=8
  163.51 images/s | 0.85 min/epoch
Running size=128, batch=16, workers=2, threads=8
  97.50 images/s | 1.42 min/epoch
Running size=160, batch=16, workers=2, threads=8
  70.40 images/s | 1.96 min/epoch


,batch_size,num_workers,img_size,cpu_threads,dataset_images,seconds_per_batch,images_per_sec,estimated_epoch_min
0,16,2,96,8,8299,0.097853,163.510700,0.845918
1,16,2,128,8,8299,0.164105,97.498509,1.418654
2,16,2,160,8,8299,0.227276,70.398891,1.964756


## 9. Benchmark D — augmentation ON vs OFF
Cho biết preprocessing augmentation có phải bottleneck hay không.


In [9]:
results_D=[]
for aug in [False,True]:
    print("Running augmentation:","ON" if aug else "OFF")
    try:
        r=benchmark_config(BEST_BATCH,BEST_WORKERS,128,BEST_THREADS,use_augmentation=aug); r["augmentation"]="ON" if aug else "OFF"; results_D.append(r)
        print(f"  {r["images_per_sec"]:.2f} images/s | {r["estimated_epoch_min"]:.2f} min/epoch")
    except Exception as e: print("  FAILED:",repr(e))
df_D=pd.DataFrame(results_D); display(df_D)


Running augmentation: OFF
  106.42 images/s | 1.30 min/epoch
Running augmentation: ON
  106.67 images/s | 1.30 min/epoch


,batch_size,num_workers,img_size,cpu_threads,dataset_images,seconds_per_batch,images_per_sec,estimated_epoch_min,augmentation
0,16,2,128,8,8299,0.150344,106.422927,1.299689,OFF
1,16,2,128,8,8299,0.150002,106.665377,1.296734,ON


## 10. Tổng hợp và xuất kết quả
Phần này là phần cần gửi lại để đánh giá cấu hình local.

In [ ]:
frames=[]
for name,df in [("A_batch_workers",df_A),("B_threads",df_B),("C_resolution",df_C),("D_augmentation",df_D)]:
    if len(df):
        z=df.copy(); z["benchmark"]=name; z=z[["benchmark"]+[c for c in z.columns if c!="benchmark"]]; frames.append(z)
df_all=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()
if len(df_all):
    display(df_all.sort_values(["benchmark","images_per_sec"],ascending=[True,False]))
    best=df_all.iloc[df_all["images_per_sec"].idxmax()]
    print("\n=== FASTEST CONFIGURATION ===")
    print("batch:",int(best.batch_size),"workers:",int(best.num_workers),"image:",int(best.img_size),"threads:",int(best.cpu_threads))
    print("throughput:",round(float(best.images_per_sec),2),"images/s")
    print("estimated epoch:",round(float(best.estimated_epoch_min),2),"min")
    out=Path("../outputs/performance_benchmark"); out.mkdir(parents=True,exist_ok=True)
    df_all.to_csv(out/"local_cpu_benchmark_results.csv",index=False)
    df_all.to_json(out/"local_cpu_benchmark_results.json",orient="records",indent=2)
    print("Saved:", (out/"local_cpu_benchmark_results.csv").resolve())


,benchmark,batch_size,num_workers,img_size,cpu_threads,dataset_images,seconds_per_batch,images_per_sec,estimated_epoch_min,augmentation
1,A_batch_workers,16,2,128,8,8299,0.173521,92.208007,1.500051,NaN
9,A_batch_workers,128,0,128,8,8299,1.413059,90.583632,1.526950,NaN
7,A_batch_workers,64,2,128,8,8299,0.708900,90.280731,1.532073,NaN
3,A_batch_workers,32,0,128,8,8299,0.367083,87.173722,1.586678,NaN
2,A_batch_workers,16,4,128,8,8299,0.183623,87.135142,1.587381,NaN
0,A_batch_workers,16,0,128,8,8299,0.190206,84.119257,1.644293,NaN
10,A_batch_workers,128,2,128,8,8299,1.546403,82.772758,1.671041,NaN
6,A_batch_workers,64,0,128,8,8299,0.774483,82.635756,1.673811,NaN
11,A_batch_workers,128,4,128,8,8299,1.617134,79.152381,1.747473,NaN
5,A_batch_workers,32,4,128,8,8299,0.405508,78.913336,1.752767,NaN



=== FASTEST CONFIGURATION ===
batch: 16 workers: 2 image: 96 threads: 8
throughput: 163.51 images/s
estimated epoch: 0.85 min
Saved: C:\Users\YOGA\Desktop\Class\05_Deep_Neural_Networks\project-01\face-anti-spoofing\outputs\performance_benchmark\local_cpu_benchmark_results.csv


## 11. Khi gửi kết quả
Gửi output của Benchmark A, B, C, D và `FASTEST CONFIGURATION`. Không cần chạy full training. Từ đó có thể chọn cấu hình batch/workers/threads/resolution phù hợp cho training thật.


In [13]:
# Final comparison: 3 candidate configurations
final_configs = [
    {"name": "A_batch32_workers2",  "batch": 32,  "workers": 2},
    {"name": "B_batch64_workers2",  "batch": 64,  "workers": 2},
    {"name": "C_batch128_workers0", "batch": 128, "workers": 0},
]

final_results = []

for cfg in final_configs:
    print(
        f"Running {cfg['name']} | "
        f"batch={cfg['batch']}, workers={cfg['workers']}, threads=8, img=128"
    )

    r = benchmark_config(
        batch_size=cfg["batch"],
        num_workers=cfg["workers"],
        img_size=128,
        cpu_threads=8,
        warmup_batches=10,
        measure_batches=100,
        use_augmentation=True
    )

    r["config"] = cfg["name"]
    final_results.append(r)

    print(
        f"  {r['images_per_sec']:.2f} images/s | "
        f"{r['estimated_epoch_min']:.2f} min/epoch"
    )

final_df = pd.DataFrame(final_results)

display(
    final_df[
        [
            "config",
            "batch_size",
            "num_workers",
            "cpu_threads",
            "img_size",
            "images_per_sec",
            "estimated_epoch_min"
        ]
    ].sort_values("images_per_sec", ascending=False)
)

Running A_batch32_workers2 | batch=32, workers=2, threads=8, img=128
  96.90 images/s | 1.43 min/epoch
Running B_batch64_workers2 | batch=64, workers=2, threads=8, img=128
  93.28 images/s | 1.48 min/epoch
Running C_batch128_workers0 | batch=128, workers=0, threads=8, img=128
  93.19 images/s | 1.48 min/epoch


,config,batch_size,num_workers,cpu_threads,img_size,images_per_sec,estimated_epoch_min
0,A_batch32_workers2,32,2,8,128,96.902399,1.427381
1,B_batch64_workers2,64,2,8,128,93.276513,1.482867
2,C_batch128_workers0,128,0,8,128,93.186154,1.484305


In [16]:
import torch

print("PyTorch:", torch.__version__)
print("XPU available:", torch.xpu.is_available())
print("XPU device count:", torch.xpu.device_count())

if torch.xpu.is_available():
    print("XPU device:", torch.xpu.get_device_name(0))

print(torch.__config__.show())

PyTorch: 2.14.0+cpu
XPU available: False
XPU device count: 0
PyTorch built with:
  - C++ Version: 202002
  - MSVC 194234444
  - Intel(R) oneAPI Math Kernel Library Version 2026.1-Product Build 20260612 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.12.0 (Git Hash 80afa71049cd69a3df32adcccb623b12cd7baa22)
  - OpenMP 2019
  - LAPACK is enabled (usually provided by MKL)
  - CPU capability usage: AVX2
  - Build settings: BLAS_INFO=mkl, BUILD_TYPE=Release, COMMIT_SHA=08187d9e0fba026dc8217405802ab5381dc88d90, CUDA_FLAGS= -DLIBCUDACXX_ENABLE_SIMPLIFIED_COMPLEX_OPERATIONS -Xcompiler /Zc:__cplusplus -Xcompiler /Zc:preprocessor -Xcompiler /w -w -Xcompiler /FS -Xfatbin -compress-all  -Xcompiler=/wd4819,/wd4503,/wd4190,/wd4244,/wd4251,/wd4275,/wd4522 -Wno-deprecated-gpu-targets --expt-extended-lambda -DCUB_WRAPPED_NAMESPACE=at_cuda_detail -DDISABLE_CUSPARSE_DEPRECATED -DCUDA_HAS_FP16=1 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ -D__